# Informe Trabajo Práctico Proyesat - Programación 1
Este notebook presenta el análisis integral del catálogo de satélites utilizando una arquitectura modular del proyecto: 'catalogo.py', 'satelite.py' y 'analisis.py', combinados con las funciones de **Skyfield**

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import catalogo
from satelite import crear_satelites
import analisis

## 1. Carga y filtrado del catálogo
Cargamos los datos desde el archivo JSON utilizando las funciones de lectura y generamos los objetos 'Satelite', descartando aquellos con una altitud menor a 100km

In [11]:
ruta_json = Path("datos/satelites.json")

registros = catalogo.leer_satelites(ruta_json)
print(f"Total de registros leídos: {len(registros)}")

momento_actual = datetime.now(timezone.utc)

satelites = crear_satelites(registros, momento_actual)
print(f"Satélites válidos analizados: {len(satelites)}")

Total de registros leídos: 16822
Satélites válidos analizados: 16822


## 2. Análisis Estadístico por gtupos y tipos de órbita
Estudiamos cómo se distribuyen los satélites y los clasificamos por el tipo de órbita (LEO, MEO, GEO, HEO)

In [12]:
conteo_grupos = catalogo.contar_por_grupo(registros)
print("Conteo de registros por grupo en el catálogo:")
for grupo_nombre, cantidad in conteo_grupos.items():
    print(f"  - {grupo_nombre}: {cantidad}")

print("\n--- Resumen Estadístico de Altitudes por Grupo ---")
resumen_grupos = analisis.resumen_por_grupo(satelites)
for grupo_nombre, stats in resumen_grupos.items():
    print(f"Grupo: {grupo_nombre}")
    print(f"  Cantidad activa: {stats['cantidad']}")
    print(f"  Altitud Mínima: {stats['minima']:.2f} km")
    print(f"  Altitud Media: {stats['media']:.2f} km")
    print(f"  Altitud Máxima: {stats['maxima']:.2f} km\n")

conteo_tipos = analisis.contar_por_tipo(satelites)
print("Distribución por tipo de órbita:")
for tipo_orb, cant in conteo_tipos.items():
    print(f"  - {tipo_orb}: {cant}")

Conteo de registros por grupo en el catálogo:
  - Otros: 4152
  - GPS: 40
  - Globalstar: 36
  - BeiDou: 55
  - Iridium: 80
  - OneWeb: 651
  - Starlink: 11160
  - Qianfan: 257
  - Kuiper: 391

--- Resumen Estadístico de Altitudes por Grupo ---
Grupo: Otros
  Cantidad activa: 4152
  Altitud Mínima: 106.11 km
  Altitud Media: 6176.97 km
  Altitud Máxima: 135980.63 km

Grupo: GPS
  Cantidad activa: 40
  Altitud Mínima: 19732.91 km
  Altitud Media: 20258.70 km
  Altitud Máxima: 21020.61 km

Grupo: Globalstar
  Cantidad activa: 36
  Altitud Mínima: 679.21 km
  Altitud Media: 1298.96 km
  Altitud Máxima: 1832.23 km

Grupo: BeiDou
  Cantidad activa: 55
  Altitud Mínima: 21512.68 km
  Altitud Media: 27487.27 km
  Altitud Máxima: 35983.96 km

Grupo: Iridium
  Cantidad activa: 80
  Altitud Mínima: 631.63 km
  Altitud Media: 777.69 km
  Altitud Máxima: 807.99 km

Grupo: OneWeb
  Cantidad activa: 651
  Altitud Mínima: 530.59 km
  Altitud Media: 1209.14 km
  Altitud Máxima: 1251.70 km

Grupo: Star

## 3. Identificación de órbitas extremas

Identificamos cuáles son los satélites que se encuentran en las posiciones más bajas y más altas del conjunto analizado

In [13]:
if satelites:
    s_bajo = analisis.mas_bajo(satelites)
    s_alto = analisis.mas_alto(satelites)
    
    print(f"Satélite más bajo: {s_bajo}")
    print(f"Satélite más alto: {s_alto}")

    print("\nTop 5 satélites más bajos:")
    top_bajos = analisis.top_n(satelites, 5, mas_altos=False)
    for s in top_bajos:
        print(f"  * {s}")
    
    print("\nTop 5 satélites más altos:")
    top_altos = analisis.top_n(satelites, 5, mas_altos=True)
    for s in top_altos:
        print(f"  * {s}")

Satélite más bajo: STARLINK-1036 | LEO | 103.831 km
Satélite más alto: CXO | HEO | 135980.629 km

Top 5 satélites más bajos:
  * STARLINK-1036 | LEO | 103.831 km
  * RASSVET-3 23 | LEO | 106.112 km
  * IPERDRONE.0 | LEO | 113.811 km
  * RASSVET-3 17 | LEO | 190.173 km
  * STARLINK-1309 | LEO | 202.426 km

Top 5 satélites más altos:
  * CXO | HEO | 135980.629 km
  * SMILE | HEO | 109940.008 km
  * THEMIS A | HEO | 73783.501 km
  * THEMIS E | HEO | 67933.975 km
  * ASTRANIS UTILITYSAT | HEO | 63409.73 km


## 4. Análisis geoespacial (filtrado por zona)

Evaluamos qué satélites se encuentran sobre una región geográfica delimitada por coordenadas de longitud y latitud `(lon_min, lat_min, lon_max, lat_max)`

In [14]:
zona_estudio = (-80.0, -60.0, -30.0, 15.0)

satelites_en_zona = analisis.en_zona(satelites, zona_estudio)
print(f"Satélites encontrados en la zona: {len(satelites_en_zona)}")
for s in satelites_en_zona[:5]:  
    print(f"  - {s} | Lat: {s.latitud:.2f}°, Lon: {s.longitud:.2f}°")

Satélites encontrados en la zona: 1350
  - TDRS 3 | GEO | 35619.649 km | Lat: 1.36°, Lon: -48.84°
  - TDRS 6 | GEO | 35815.404 km | Lat: -1.65°, Lon: -45.55°
  - INMARSAT 3-F5 | GEO | 35799.468 km | Lat: -9.42°, Lon: -53.64°
  - SCD 2 | LEO | 734.484 km | Lat: 12.98°, Lon: -44.78°
  - USA 149 (DSP 20) | GEO | 36135.112 km | Lat: -9.40°, Lon: -68.53°


## 5. Conclusiones

* La modularización del código facilitó la separación de responsabilidades: la lectura de archivos y clasificación en `catalogo.py`, el modelo físico y geométrico en `satelite.py`, y las métricas avanzadas en `analisis.py`
  
* Las constelaciones comerciales en órbita baja (LEO), como Starlink, dominan ampliamente la distribución numérica del catálogo actual

  
* El filtrado inicial de objetos menores a 100 km garantiza que el análisis orbital no se vea distorsionado por reentradas atmosféricas o artefactos numéricos